In [1]:
import os
from termcolor import cprint
HERE = 'c:\\Users\\harry\\Documents\\code_ws\\pb_ws\\coop_assembly\\src\\coop_assembly\\planning\\'
pyplanner_path = os.path.abspath(os.path.join(HERE, '..', '..', '..', 'external', 'pyplanners'))
sys.path.extend([
    os.path.abspath(os.path.join(HERE, '..', '..', '..', 'external', 'pddlstream')),
    pyplanner_path,
])
os.environ['PYPLANNERS_PATH'] = pyplanner_path

import pddlstream
cprint("Using pddlstream from {}".format(pddlstream.__file__), 'yellow')
import strips # pyplanners
cprint("Using strips from {}".format(strips.__file__), 'yellow')

Using pddlstream from c:\Users\harry\Documents\code_ws\pb_ws\coop_assembly\external\pddlstream\pddlstream\__init__.py
Using strips from c:\Users\harry\Documents\code_ws\pb_ws\coop_assembly\external\pyplanners\strips\__init__.py


In [2]:
import numpy as np
np.set_printoptions(precision=3)

## Load shapes

In [3]:
test_file_name = '1_exp_tets.json'

In [4]:
from coop_assembly.data_structure import BarStructure
from coop_assembly.planning.parsing import get_assembly_path
from coop_assembly.help_functions.parsing import parse_saved_structure_data
from coop_assembly.help_functions.shared_const import METER_SCALE

b_struct_data, o_struct_data = parse_saved_structure_data(get_assembly_path(test_file_name))
if 'data' in b_struct_data:
    b_struct_data = b_struct_data['data']
bar_struct = BarStructure.from_data(b_struct_data)

Parsed file name: c:\users\harry\documents\code_ws\pb_ws\coop_assembly\tests\test_data\1_exp_tets.json | write_time: 2020-12-15 19:42:09.183482 | 


In [5]:
from coop_assembly.planning.parsing import unpack_structure

element_from_index, grounded_elements, contact_from_connectors, connectors = \
    unpack_structure(bar_struct, chosen_bars=None, scale=METER_SCALE)

## Define stream functions

In [44]:
from coop_assembly.planning.utils import compute_z_distance, check_connected, Command

def get_wild_remove_element_gen_fn(element_from_index, grounded_elements, connectors, debug=False, use_fluent=True, \
    connectivity=True):
    def wild_gen_fn(robot_name, element, fluents=[]):
        next_printed = set()
        for fact in fluents:
            if fact[0] == 'assembled':
                #if fact[1] != element:
                next_printed.add(fact[1])
            else:
                raise NotImplementedError(fact[0])

        print('---')
        assert element in next_printed
        next_printed = next_printed - {element}
        print('Remove E{} - fluent printed ({}): {}'.format(element, len(next_printed), sorted(next_printed)))

        # # * partial ordering
        # if ordering and (outgoing_from_element[element] & next_printed):
        #     # outgoing elements must be removed BEFORE element
        #     cprint('Ordering fails.', 'red')
        #     return

        # * connectivity
        if connectivity and not check_connected(connectors, grounded_elements, next_printed):
            cprint('Connectivity fails.', 'red')
            return

        # # * stiffness
        # if stiffness and not check_connected(connectors, grounded_elements, next_printed):
        #     # TODO: stiffness
        #     return

        if debug:
            num_outputs = 2 if use_fluent else 3
            outputs = [num_outputs*(None,)]
            yield outputs # For debugging
            return
        
        command = Command([])
        grasp = None
        outputs = [(command, grasp)]
        yield outputs
    # end wild_gen_fn
    return wild_gen_fn

## Construct PDDL problem

In [45]:
printed=set()
removed=set()
debug = False

In [46]:
from pddlstream.utils import read
from pddlstream.language.stream import StreamInfo, PartialInputs, WildOutput, DEBUG
from pddlstream.language.constants import And, PDDLProblem, Equal, print_plan

domain_pddl = read(os.path.join(HERE, 'pddl/auto_scaffolds/domain.pddl'))
stream_pddl = read(os.path.join(HERE, 'pddl/auto_scaffolds/stream.pddl'))

init = []
init.extend([
    ('Robot', 'r0'),
])

remaining = set(element_from_index) - removed - printed
for e in remaining:
    init.extend([
        ('Element', e),
        ('Assembled', e),
    ])

constant_map = {}
stream_map = {
    'sample-remove': get_wild_remove_element_gen_fn(element_from_index, grounded_elements, connectors, debug=debug)
}
if debug:
    stream_map = DEBUG

goal_literals = []
goal_literals.extend(('Removed', e) for e in remaining)
goal = And(*goal_literals)

pddlstream_problem = PDDLProblem(domain_pddl, constant_map, stream_pddl, stream_map, init, goal)

In [47]:
print('Init:', pddlstream_problem.init)
print('Goal:', pddlstream_problem.goal)

Init: [('Robot', 'r0'), ('Element', 0), ('Assembled', 0), ('Element', 1), ('Assembled', 1), ('Element', 2), ('Assembled', 2), ('Element', 3), ('Assembled', 3), ('Element', 4), ('Assembled', 4), ('Element', 5), ('Assembled', 5)]
Goal: ('and', ('Removed', 0), ('Removed', 1), ('Removed', 2), ('Removed', 3), ('Removed', 4), ('Removed', 5))


## Solve

In [48]:
costs = False

In [49]:
from pddlstream.algorithms.downward import set_cost_scale, parse_action
from pddlstream.algorithms.incremental import solve_incremental
from pddlstream.utils import INF

set_cost_scale(1)
success_cost = 0 if costs else INF
discrete_planner = {
    'search': 'eager', # eager | lazy
    # lazy might be faster but the solution quality might be low
    'evaluator': 'greedy',
    'heuristic': 'goal', # goal | add | ff
    #'heuristic': ['ff', get_bias_fn(element_from_index)],
    'successors': 'all', # all | random
    #'successors': get_order_fn(element_from_index),
    # ? ordering scaffolds can come here
    # TODO: confirm that this is working correctly
}
solution = solve_incremental(pddlstream_problem, max_time=600, planner=discrete_planner,
                                success_cost=success_cost, unit_costs=not costs,
                                max_planner_time=300, debug=debug, verbose=True)

Calls: 0 | Successes: 0 | Results: 0 | Counts: Counter()
Iteration: 1 | Complexity: 0 | Calls: 0 | Evaluations: 13 | Solved: False | Cost: inf | Time: 0.001
---
Remove E0 - fluent printed (5): [1, 2, 3, 4, 5]
---
Remove E1 - fluent printed (4): [2, 3, 4, 5]
---
Remove E2 - fluent printed (3): [3, 4, 5]
Connectivity fails.
---
Remove E3 - fluent printed (3): [2, 4, 5]
---
Remove E3 - fluent printed (2): [4, 5]
Connectivity fails.
---
Remove E4 - fluent printed (2): [2, 5]
---
Remove E4 - fluent printed (1): [5]
Connectivity fails.
---
Remove E5 - fluent printed (1): [2]
---
Remove E5 - fluent printed (0): []
Summary: {complexity: 0, cost: 6.0, evaluations: 13, iterations: 1, length: 6, run_time: 0.01894521713256836, solutions: 1, solved: True, timeout: False}


In [52]:
from pddlstream.language.constants import print_plan, is_plan

plan, _, facts = solution
print('-'*10)
print_plan(plan)
assert is_plan(plan)

print('='*10)
place_actions = [action for action in reversed(plan) if action.name == 'remove_element']
commands = []

----------
 1) remove_element r0 c[] 0 None
 2) remove_element r0 c[] 1 None
 3) remove_element r0 c[] 3 None
 4) remove_element r0 c[] 4 None
 5) remove_element r0 ~ 2 ~
 6) remove_element r0 c[] 5 None
